In [1]:

import pandas as pd
import fasttext.util
import nltk
import sys
import os
import re
import numpy as np
import networkx as nx
from textblob_de import TextBlobDE
from sklearn.metrics.pairwise import cosine_similarity

os.chdir(r"C:\Users\Giuliano.DESKTOP-NPATJ24\Desktop\ptsd_classifier_new_code")
sys.path.append(os.path.abspath(r"C:\Users\Giuliano.DESKTOP-NPATJ24\Desktop\ptsd_classifier_new_code"))

from src.functions import (
    analyze_text_responses,
    score_questionnaire,
    generate_full_text_embed,
    compute_total_semvar_over_responses,
    compute_pairwise_semantic_distances
)
from src.models import load_models
model, model_word, tokenizer_bert, model_bert = load_models()



dataset = pd.read_csv(
    "data/Online-Erhebung+Giuliano+Trauma-Abfrage_October+9,+2025_12.14_new.csv",
    sep=",",
    decimal=".",
    encoding="utf-8"
)

textvariables = [
    "response_negnt",
    "response_neutr",
    "response_trauma"
]

c:\Users\Giuliano.DESKTOP-NPATJ24\anaconda3\envs\emb_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Participant exclusion

In [2]:
dataset = dataset.iloc[2:,:]
dataset = dataset[dataset['Finished'] == "1"]
dataset = dataset.reset_index(drop=True)

response_cols = [f"response_{i}" for i in ["trauma", "negnt", "neutr"]]
dataset = dataset.dropna(subset=response_cols)

vp_letters = ["vpcode_1", "vpcode_2"]
vp_numbers = ["vpcode_3", "vpcode_4"]
dataset[vp_letters] = dataset[vp_letters].apply(lambda x: x.str.lower() if x.dtype == "object" else x) #alle strings zu lower case machen
dataset[vp_numbers] = dataset[vp_numbers].map(lambda x: re.sub(r'\b(\d)\b', r'0\1', str(x))) #alle Einzelziffern "X" ohne 0 davor zu "0X" umwandeln
vp_cols = ["vpcode_1", "vpcode_2", "vpcode_3", "vpcode_4"]
dataset = dataset[~dataset.duplicated(subset=vp_cols, keep="first")]
print(len(dataset))
#Auschluss von Leuten deren Antworten Schwachsinn oder AI enthielten
dataset = dataset[dataset.index != 13].reset_index(drop=True)
dataset = dataset[dataset.index != 98].reset_index(drop=True)
dataset = dataset[~dataset.index.isin([35, 36, 92, 123])].reset_index(drop=True)
dataset = dataset[dataset["ResponseId"] != "R_8rIcX3F7sMpntHu"]
#Ausschluss von Leuten, welche die Instruktionen laut eigener Angabe nicht verstanden haben
dataset = dataset[~dataset.index.isin([64,98,118])].reset_index(drop=True)
print(len(dataset))


169
159


Analyse text responses & exclude participants with less than 20 words or 2 sentences.


In [3]:
analyze_text_responses(dataset, textvariables, model, model_word, tokenizer_bert, model_bert)
dataset = dataset[
    (dataset["response_negnt_wc"] >= 20) &
    (dataset["response_neutr_wc"] >= 20) &
    (dataset["response_trauma_wc"] >= 20) &
    (dataset["response_negnt_sc"] > 1) &
    (dataset["response_neutr_sc"] > 1) &
    (dataset["response_trauma_sc"] > 1)
]

c:\Users\Giuliano.DESKTOP-NPATJ24\anaconda3\envs\emb_env\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
C:\Users\Giuliano.DESKTOP-NPATJ24\Desktop\ptsd_classifier_new_code\src\functions.py:85: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset.loc[i, f"{column_name}_{k}_order_static_word_coherence_max"] = np.max(diag)
c:\Users\Giuliano.DESKTOP-NPATJ24\anaconda3\envs\emb_env\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwarg

Questionnaire scoring

In [4]:
#scoring
dataset = score_questionnaire(dataset, "PCL-5", 20, "PCL-5_total")
dataset = score_questionnaire(dataset, "dass21", 21, "dass21_total")
dataset = score_questionnaire(dataset, "swe", 10, "swe_total")

dataset = dataset.reset_index(drop=True)

#put into conditions
dataset.loc[dataset["PCL-5_total"] >= 34, "condition_pcl"] = "clin_pcl"
dataset.loc[dataset["PCL-5_total"] < 34, "condition_pcl"] = "subclinical"
dataset["DASS21_stress"] = (dataset[["dass21_1","dass21_6","dass21_8","dass21_11","dass21_12","dass21_14","dass21_18"]].sum(axis=1))*2
dataset["DASS21_anxiety"] = (dataset[["dass21_2","dass21_4","dass21_7","dass21_9","dass21_15","dass21_19","dass21_20"]].sum(axis=1))*2
dataset["DASS21_depression"] = (dataset[["dass21_3","dass21_5","dass21_10","dass21_13","dass21_16","dass21_17","dass21_21"]].sum(axis=1))*2

dataset.loc[dataset["DASS21_stress"] >= 19, "condition_dass_stress"] = "clin_dass_stress"
dataset.loc[dataset["DASS21_stress"] < 19, "condition_dass_stress"] = "subclinical_dass_stress"
dataset.loc[dataset["DASS21_anxiety"] >= 10, "condition_dass_anxiety"] = "clin_dass_anxiety"
dataset.loc[dataset["DASS21_anxiety"] < 10, "condition_dass_anxiety"] = "subclinical_dass_anxiety"
dataset.loc[dataset["DASS21_depression"] >= 14, "condition_dass_depression"] = "clin_dass_depression"
dataset.loc[dataset["DASS21_depression"] < 14, "condition_dass_depression"] = "subclinical_dass_depression"

C:\Users\Giuliano.DESKTOP-NPATJ24\Desktop\ptsd_classifier_new_code\src\functions.py:127: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset[total_name] = dataset[columns].sum(axis=1)
C:\Users\Giuliano.DESKTOP-NPATJ24\Desktop\ptsd_classifier_new_code\src\functions.py:127: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset[total_name] = dataset[columns].sum(axis=1)
C:\Users\Giuliano.DESKTOP-NPATJ24\Desktop\ptsd_classifier_new_code\src\functions.py:127: PerformanceWarning: DataFrame is highly fragmented.  This is usually the 

Generate embeddings for the entire texts

In [5]:
generate_full_text_embed(dataset, textvariables, model)


c:\Users\Giuliano.DESKTOP-NPATJ24\anaconda3\envs\emb_env\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
C:\Users\Giuliano.DESKTOP-NPATJ24\Desktop\ptsd_classifier_new_code\src\functions.py:141: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset.loc[i, f"{column_name}_embedding_{dim_idx}"] = value
C:\Users\Giuliano.DESKTOP-NPATJ24\Desktop\ptsd_classifier_new_code\src\functions.py:141: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1

,StartDate,EndDate,Status,Progress,Duration (in seconds),Finished,RecordedDate,ResponseId,DistributionChannel,UserLanguage,...,response_trauma_embedding_1014,response_trauma_embedding_1015,response_trauma_embedding_1016,response_trauma_embedding_1017,response_trauma_embedding_1018,response_trauma_embedding_1019,response_trauma_embedding_1020,response_trauma_embedding_1021,response_trauma_embedding_1022,response_trauma_embedding_1023
0,2025-01-10 19:08:23,2025-01-10 19:42:30,0,100,2047,1,2025-01-10 19:42:31,R_83or6ZudLiBqawF,anonymous,EN,...,0.132775,0.191113,0.131074,0.066128,-0.054176,-0.351291,-0.119335,0.082426,0.301551,-0.527962
1,2025-01-10 16:30:29,2025-01-10 21:31:48,0,100,18079,1,2025-01-10 21:31:49,R_83qLClUC1bexEdj,anonymous,EN,...,0.199266,0.139232,-0.033862,0.222106,-0.026325,-0.288911,-0.145985,0.070614,0.204733,-0.554780
2,2025-01-11 12:29:09,2025-01-11 13:26:51,0,100,3462,1,2025-01-11 13:26:52,R_2pu9u27pGywz0yC,anonymous,EN,...,0.250138,0.166890,0.055059,0.087417,-0.079755,-0.191903,-0.172914,0.175738,0.334309,-0.437064
3,2025-01-11 12:12:19,2025-01-11 13:46:53,0,100,5674,1,2025-01-11 13:46:54,R_2mmB2zG3eESUbPd,anonymous,EN,...,0.383705,0.250134,0.074668,0.005780,0.041227,-0.325879,-0.229743,0.312908,0.147091,-0.504463
4,2025-01-11 19:29:04,2025-01-11 20:18:13,0,100,2948,1,2025-01-11 20:18:13,R_2kNQQoiJfR6in3O,anonymous,EN,...,0.061862,0.218824,0.096460,0.235049,0.012873,-0.296395,-0.126989,0.141027,0.142072,-0.419886
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,2025-07-28 10:43:18,2025-07-28 11:30:50,0,100,2852,1,2025-07-28 11:30:50,R_8jYTXI82lJgYmkX,anonymous,EN,...,0.237765,0.176755,0.126163,0.153190,-0.040521,-0.180671,-0.149060,0.184327,0.081391,-0.387566
146,2025-07-28 12:14:43,2025-07-28 12:55:45,0,100,2462,1,2025-07-28 12:55:46,R_2V9dahU04CAIJjN,anonymous,EN,...,0.272246,0.254408,-0.117896,-0.054259,-0.048607,-0.294707,-0.234865,0.146070,0.172241,-0.594467
147,2025-07-28 20:19:28,2025-07-28 21:02:17,0,100,2569,1,2025-07-28 21:02:18,R_2L9KykKJnFmIBR7,anonymous,EN,...,0.248608,0.146290,0.013959,-0.011694,0.057198,-0.223357,-0.087169,0.059058,0.116206,-0.496829
148,2025-07-30 12:01:22,2025-07-30 12:38:44,0,100,2241,1,2025-07-30 12:38:45,R_2qX8BxWOTEO5xNh,anonymous,EN,...,0.240017,0.103639,0.137186,0.028230,-0.128039,-0.218193,-0.130501,0.082521,0.276183,-0.426955


In [6]:
compute_total_semvar_over_responses(dataset, textvariables, model)

c:\Users\Giuliano.DESKTOP-NPATJ24\Desktop\PTSD_online_survey\src\functions.py:175: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset.at[i, "total_semvar_over_responses"] = mean_var


,StartDate,EndDate,Status,Progress,Duration (in seconds),Finished,RecordedDate,ResponseId,DistributionChannel,UserLanguage,...,response_trauma_embedding_375,response_trauma_embedding_376,response_trauma_embedding_377,response_trauma_embedding_378,response_trauma_embedding_379,response_trauma_embedding_380,response_trauma_embedding_381,response_trauma_embedding_382,response_trauma_embedding_383,total_semvar_over_responses
0,2025-01-10 19:08:23,2025-01-10 19:42:30,0,100,2047,1,2025-01-10 19:42:31,R_83or6ZudLiBqawF,anonymous,EN,...,0.170930,-0.365130,0.217697,-0.293684,-0.042687,0.257298,0.142614,-0.351227,-0.208413,0.012971
1,2025-01-10 16:30:29,2025-01-10 21:31:48,0,100,18079,1,2025-01-10 21:31:49,R_83qLClUC1bexEdj,anonymous,EN,...,0.003976,0.101481,0.116883,-0.127257,-0.049332,0.011288,-0.017677,0.069838,-0.055203,0.015139
2,2025-01-11 12:29:09,2025-01-11 13:26:51,0,100,3462,1,2025-01-11 13:26:52,R_2pu9u27pGywz0yC,anonymous,EN,...,-0.091919,0.060585,-0.122786,-0.192184,-0.192840,0.078944,0.185225,-0.260592,-0.136543,0.013643
3,2025-01-11 12:12:19,2025-01-11 13:46:53,0,100,5674,1,2025-01-11 13:46:54,R_2mmB2zG3eESUbPd,anonymous,EN,...,-0.266977,-0.165408,0.104277,-0.120912,-0.141919,0.270893,0.430351,-0.085916,-0.007261,0.011658
4,2025-01-11 19:29:04,2025-01-11 20:18:13,0,100,2948,1,2025-01-11 20:18:13,R_2kNQQoiJfR6in3O,anonymous,EN,...,0.388776,0.068313,0.171024,0.014622,-0.058713,0.258412,-0.049455,-0.014994,-0.246332,0.012412
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,2025-07-28 10:43:18,2025-07-28 11:30:50,0,100,2852,1,2025-07-28 11:30:50,R_8jYTXI82lJgYmkX,anonymous,EN,...,-0.294467,-0.105197,0.090523,-0.074875,-0.075463,0.344547,0.256669,-0.061250,-0.104550,0.012092
146,2025-07-28 12:14:43,2025-07-28 12:55:45,0,100,2462,1,2025-07-28 12:55:46,R_2V9dahU04CAIJjN,anonymous,EN,...,-0.197991,-0.126130,0.173415,-0.072077,0.107336,0.228160,0.125365,-0.120192,-0.054084,0.010766
147,2025-07-28 20:19:28,2025-07-28 21:02:17,0,100,2569,1,2025-07-28 21:02:18,R_2L9KykKJnFmIBR7,anonymous,EN,...,-0.269836,-0.011668,0.087713,-0.136749,-0.105770,0.396010,0.238228,0.146688,-0.152024,0.014659
148,2025-07-30 12:01:22,2025-07-30 12:38:44,0,100,2241,1,2025-07-30 12:38:45,R_2qX8BxWOTEO5xNh,anonymous,EN,...,-0.339893,-0.045579,0.144323,0.028749,-0.053724,0.113800,0.249172,-0.126821,0.095626,0.013850


In [7]:
compute_pairwise_semantic_distances(dataset, textvariables, model)

c:\Users\Giuliano.DESKTOP-NPATJ24\Desktop\PTSD_online_survey\src\functions.py:202: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset.at[i, colname] = np.nan
c:\Users\Giuliano.DESKTOP-NPATJ24\Desktop\PTSD_online_survey\src\functions.py:202: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset.at[i, colname] = np.nan
c:\Users\Giuliano.DESKTOP-NPATJ24\Desktop\PTSD_online_survey\src\functions.py:202: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has p

,StartDate,EndDate,Status,Progress,Duration (in seconds),Finished,RecordedDate,ResponseId,DistributionChannel,UserLanguage,...,response_trauma_embedding_378,response_trauma_embedding_379,response_trauma_embedding_380,response_trauma_embedding_381,response_trauma_embedding_382,response_trauma_embedding_383,total_semvar_over_responses,dist_response_negnt_response_neutr,dist_response_negnt_response_trauma,dist_response_neutr_response_trauma
0,2025-01-10 19:08:23,2025-01-10 19:42:30,0,100,2047,1,2025-01-10 19:42:31,R_83or6ZudLiBqawF,anonymous,EN,...,-0.293684,-0.042687,0.257298,0.142614,-0.351227,-0.208413,0.012971,0.751855,0.769760,0.555016
1,2025-01-10 16:30:29,2025-01-10 21:31:48,0,100,18079,1,2025-01-10 21:31:49,R_83qLClUC1bexEdj,anonymous,EN,...,-0.127257,-0.049332,0.011288,-0.017677,0.069838,-0.055203,0.015139,0.566328,0.768328,0.730290
2,2025-01-11 12:29:09,2025-01-11 13:26:51,0,100,3462,1,2025-01-11 13:26:52,R_2pu9u27pGywz0yC,anonymous,EN,...,-0.192184,-0.192840,0.078944,0.185225,-0.260592,-0.136543,0.013643,0.738677,0.713399,0.609366
3,2025-01-11 12:12:19,2025-01-11 13:46:53,0,100,5674,1,2025-01-11 13:46:54,R_2mmB2zG3eESUbPd,anonymous,EN,...,-0.120912,-0.141919,0.270893,0.430351,-0.085916,-0.007261,0.011658,0.553776,0.617421,0.580211
4,2025-01-11 19:29:04,2025-01-11 20:18:13,0,100,2948,1,2025-01-11 20:18:13,R_2kNQQoiJfR6in3O,anonymous,EN,...,0.014622,-0.058713,0.258412,-0.049455,-0.014994,-0.246332,0.012412,0.854296,0.809178,0.759916
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,2025-07-28 10:43:18,2025-07-28 11:30:50,0,100,2852,1,2025-07-28 11:30:50,R_8jYTXI82lJgYmkX,anonymous,EN,...,-0.074875,-0.075463,0.344547,0.256669,-0.061250,-0.104550,0.012092,0.745502,0.526077,0.861798
146,2025-07-28 12:14:43,2025-07-28 12:55:45,0,100,2462,1,2025-07-28 12:55:46,R_2V9dahU04CAIJjN,anonymous,EN,...,-0.072077,0.107336,0.228160,0.125365,-0.120192,-0.054084,0.010766,0.640752,0.412459,0.375299
147,2025-07-28 20:19:28,2025-07-28 21:02:17,0,100,2569,1,2025-07-28 21:02:18,R_2L9KykKJnFmIBR7,anonymous,EN,...,-0.136749,-0.105770,0.396010,0.238228,0.146688,-0.152024,0.014659,0.722822,0.771620,0.759855
148,2025-07-30 12:01:22,2025-07-30 12:38:44,0,100,2241,1,2025-07-30 12:38:45,R_2qX8BxWOTEO5xNh,anonymous,EN,...,0.028749,-0.053724,0.113800,0.249172,-0.126821,0.095626,0.013850,0.596694,0.623009,0.989698


Safe dataset

In [6]:
dataset.to_csv(
    "data/dataset_preprocessed.csv",
    sep=";",
    header=True,
    decimal=".",
    encoding="utf-8",
    index=False
)